# search_similar 유사도 스코어 확인

`vs_method.py` (metadata JSONB 버전) 로 적재 → 검색 결과의 **cosine 유사도**를 표/그래프로 확인하는 테스트 노트북.

- 스코어 = `1 - (embedding <=> query)` (1.0 에 가까울수록 관련)
- 준비물(.env): `DB_URL`, `OPENAI_API_KEY`


## 1. 경로 · 환경 · import

In [1]:
import os, sys

# notebooks/ 어디에 있든 src/core 를 위로 탐색
def find_src_core(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        cand = os.path.join(d, "src", "core")
        if os.path.isfile(os.path.join(cand, "vs_method.py")):
            return cand
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None

SRC = find_src_core()
assert SRC, "src/core/vs_method.py 를 찾지 못했습니다. 노트북 위치를 확인하세요."
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print("src/core:", SRC)

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())          # 상위 폴더의 .env 도 탐색
assert os.getenv("DB_URL"),        ".env 에 DB_URL 필요"
assert os.getenv("OPENAI_API_KEY"), ".env 에 OPENAI_API_KEY 필요"

import vs_method
print("EMBEDDING:", vs_method.EMBEDDING_MODEL, "/ dim", vs_method.EMBEDDING_DIM)


src/core: c:\Users\playdata2\workspace\SKN30-3rd-4Team-dev\src\core
EMBEDDING: text-embedding-3-small / dim 1024


## 2. 연결 + 스키마

In [2]:
conn = vs_method.get_conn()
vs_method.ensure_schema(conn)
print("연결 · 스키마 OK")


연결 · 스키마 OK


## 3. 샘플 적재 (재실행 시 초기화)

`RESEED=True` 면 테이블을 비우고 다시 넣어 **중복 없이** 스코어를 확인할 수 있다.
실데이터로 이미 채웠다면 `RESEED=False` 로 두고 이 셀은 건너뛴다.

In [ ]:
RESEED = True

if RESEED:
    vs_method.clear_table(conn)

    # 법령 (statute → authority=binding)
    vs_method.ingest_document(conn,
        "제3조(대항력 등) ① 임대차는 그 등기가 없는 경우에도 임차인이 주택의 인도와 "
        "주민등록을 마친 때에는 그 다음 날부터 제삼자에 대하여 효력이 생긴다.\n"
        "제8조(보증금 중 일정액의 보호) ① 임차인은 보증금 중 일정액을 다른 담보물권자보다 "
        "우선하여 변제받을 권리가 있다.",
        {"source_type":"statute","source_org":"법제처","doc_title":"주택임대차보호법",
         "doc_year":2023,"stage":"both","issue":["deposit","opposing_power","priority_repayment"],
         "law_name":"주택임대차보호법"},
        split_preset="law")

    # 판례 (precedent → binding, court/case_no 키)
    vs_method.ingest_document(conn,
        "대항요건과 확정일자를 갖춘 임차인은 후순위 담보권자보다 우선하여 보증금을 변제받을 수 있다. "
        "주택의 인도와 주민등록은 대항력의 존속요건이다.",
        {"source_type":"precedent","source_org":"대법원","doc_title":"대법원 2013다12345 판결",
         "doc_year":2014,"stage":"post","issue":["deposit","priority_repayment"],
         "court":"대법원","case_no":"2013다12345","선고일":"2014-03-27"},
        split_preset="default")

    # 상담사례 (counsel_case → persuasive)
    vs_method.ingest_document(conn,
        "사례) 임대인이 계약 만료 후에도 보증금을 돌려주지 않는 경우, 임차권등기명령을 신청하고 "
        "내용증명을 발송한 뒤 보증금반환청구소송으로 진행할 수 있다.",
        {"source_type":"counsel_case","source_org":"한국부동산원",
         "doc_title":"2024 주택임대차 상담사례집","doc_year":2024,
         "stage":"post","issue":["deposit"]},
        split_preset="case")

    print("적재 완료")

# 적재량 확인
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM kb_chunks;")
    print("총 청크 수:", cur.fetchone()[0])


## 4. 스코어 확인 헬퍼

In [3]:
import pandas as pd

def cite(r):
    st = r.get("source_type")
    if st == "precedent":
        return " ".join(x for x in (r.get("court"), r.get("case_no")) if x) or r.get("doc_title","")
    head = r.get("law_name") or r.get("doc_title","")
    return " ".join(x for x in (head, r.get("article")) if x)

def search_df(query, **kw):
    """search_similar 결과를 스코어 내림차순 표로."""
    hits = vs_method.search_similar(conn, query, **kw)
    df = pd.DataFrame([{
        "score":     h["similarity"],
        "source":    h.get("source_type"),
        "authority": h.get("authority"),
        "cite":      cite(h),
        "snippet":   h["content"][:45].replace("\n"," ") + "…",
    } for h in hits])
    return df


## 5. 검색 예시 — 스코어 표

In [4]:
# (1) 필터 없이
search_df("전세 보증금 못 받을 때 우선변제 받는 방법", k=8)


,score,source,authority,cite,snippet
0,0.4029,statute,binding,주택임대차보호법 제3조,제3조(대항력 등) ① 임대차는 그 등기가 없는 경우에도 임차인이 주택의 인도와 …
1,0.3823,precedent,binding,대법원 2013다12345,대항요건과 확정일자를 갖춘 임차인은 후순위 담보권자 기타 채권자보다 우선하여 보증…
2,0.2158,mediation_case,persuasive,2021 주택·상가건물 임대차분쟁조정사례집,Ⅰ. 제도 개요 ① 목적 - 주택 임대차계약 관계에서 발생되는 각종 분쟁에 대하…
3,0.2100,mediation_case,persuasive,2021 주택·상가건물 임대차분쟁조정사례집,② 연혁 및 설치근거 - (연혁) | 시기 | 내용 …


In [5]:
# (2) 계약 전 + 대항력 쟁점
search_df("대항력을 갖추려면 어떻게 해야 하나", stage="pre", issues=["opposing_power"], k=5)


,score,source,authority,cite,snippet
0,0.3208,statute,binding,주택임대차보호법 제3조,제3조(대항력 등) ① 임대차는 그 등기가 없는 경우에도 임차인이 주택의 인도와 …


In [ ]:
# (3) 결론 근거만 (authority=binding)
search_df("보증금 우선변제", authorities=["binding"], k=5)


In [ ]:
# (4) 판례만 + 임의 키 필터
search_df("보증금 우선변제 판례", source_types=["precedent"], meta_filter={"court":"대법원"}, k=5)


## 6. 스코어 분포 시각화 (선택)

In [ ]:
import matplotlib.pyplot as plt

QUERY = "집주인이 보증금을 안 돌려줘요"
df = search_df(QUERY, k=8)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(range(len(df)), df["score"])
ax.invert_yaxis()
# 한글 폰트 미설정 시 라벨이 깨질 수 있어 rank+authority(영문)로 표기
ax.set_yticks(range(len(df)))
ax.set_yticklabels([f"#{i} ({a})" for i, a in enumerate(df["authority"])])
ax.set_xlim(0, 1)
ax.set_xlabel("cosine similarity")
ax.set_title(f"top-{len(df)} scores")
for i, s in enumerate(df["score"]):
    ax.text(s + 0.01, i, f"{s:.3f}", va="center", fontsize=9)
plt.tight_layout(); plt.show()
df[["score","source","cite"]]


## 7. 파라미터 바꿔가며 확인

아래 값만 바꿔 재실행하면 스코어를 바로 비교할 수 있다.
`min_score` 를 올리면 노이즈 컷 효과를, `k` 를 늘리면 하위 스코어까지 볼 수 있다.

In [ ]:
QUERY      = "보일러 수리비는 누가 부담하나요?"
K          = 8
MIN_SCORE  = 0.0          # 예: 0.35 로 올려 노이즈 컷 확인
STAGE      = None          # "pre" | "post" | None
ISSUES     = None          # 예: ["repair"]
AUTHORITIES = None         # 예: ["binding"]

search_df(QUERY, stage=STAGE, issues=ISSUES, authorities=AUTHORITIES,
          k=K, min_score=MIN_SCORE)


In [ ]:
# 최상위 히트의 전체 metadata 키 확인 (문서마다 키가 다름)
hits = vs_method.search_similar(conn, QUERY, k=1)
hits[0] if hits else "결과 없음"
